In [1]:
import pandas as pd
import os

# ================= 配置区 =================
KG_FILE = "kg.csv"  # PrimeKG 原始文件路径

# 针对 NACC 数据集定制的侦察组
# 我们需要确认这些词在 PrimeKG 里是否存在，以及它们是否关联了太多的通用节点
SEARCH_GROUPS = {
    "【核心-AD与认知】": [
        "Alzheimer", "Dementia", "Memory", "Cognitive", "Amyloid", "Tau", "Neurofibrillary"
    ],
    "【NACC-心血管与代谢】": [
        "Hypertension", "Cholesterol", "Lipid", "Hyperlipidemia", 
        "Diabetes", "Insulin", "Glucose", "Stroke", "Cerebrovascular",
        "Atrial fibrillation", "Heart failure"
    ],
    "【NACC-精神与其它】": [
        "Depression", "Anxiety", "Thyroid"
    ]
}

def main():
    print(f"🚀 [Step 1] 正在加载 {KG_FILE} 进行 NACC 适配侦察...")
    
    try:
        # 只读取必要列以节省内存
        df = pd.read_csv(KG_FILE, usecols=['relation', 'x_name', 'x_type', 'y_name', 'y_type'], low_memory=False)
    except ValueError:
        df = pd.read_csv(KG_FILE, low_memory=False)
        
    print(f"✅ PrimeKG 加载完成，共 {len(df)} 条数据。\n")

    # 1. 关键词命中测试
    print("="*60)
    print("🔎 1. 关键词命中测试 (确认 NACC 特征能否映射到生物图谱)")
    print("="*60)
    
    # 剔除药物和暴露 (我们要找的是生物学机制)
    mask_bio = (~df['x_type'].isin(['drug', 'exposure'])) & (~df['y_type'].isin(['drug', 'exposure']))
    df_bio = df[mask_bio]

    found_keywords = set()

    for group, keywords in SEARCH_GROUPS.items():
        print(f"\n--- {group} ---")
        pattern = '|'.join(keywords)
        
        # 模糊匹配
        mask = df_bio['x_name'].astype(str).str.contains(pattern, case=False, na=False) | \
               df_bio['y_name'].astype(str).str.contains(pattern, case=False, na=False)
        subset = df_bio[mask]
        
        if len(subset) > 0:
            print(f"   ✅ 命中 {len(subset)} 条相关关系")
            # 提取具体的节点名，方便后续确认
            related_nodes = set(subset['x_name']) | set(subset['y_name'])
            # 简单的过滤，只看包含关键词的节点
            hits = [n for n in related_nodes if any(k.lower() in str(n).lower() for k in keywords)]
            print(f"   📌 典型节点示例: {hits[:5]}")
            found_keywords.update(keywords)
        else:
            print(f"   ❌ 警告：未找到相关节点！请检查拼写或同义词。")

    # 2. 黑名单侦察 (Hub Node Detection)
    print("\n" + "="*60)
    print("💣 2. 黑名单侦察 (找出连接数过高且无特异性的干扰节点)")
    print("="*60)
    print("   说明：以下节点如果在 Top 20 且是通用解剖部位(如 kidney, lung)或通用概念，")
    print("   务必添加到下一步的 BLACKLIST_NODES 中！\n")

    # 统计所有节点的度 (Degree)
    all_nodes = pd.concat([df_bio['x_name'], df_bio['y_name']])
    node_counts = all_nodes.value_counts().head(30)

    print(f"{'Rank':<5} | {'Node Name':<40} | {'Count':<10}")
    print("-" * 60)
    for i, (node, count) in enumerate(node_counts.items()):
        print(f"{i+1:<5} | {node:<40} | {count:<10}")

    print("\n✅ [Step 1] 侦察完成！请根据上述结果：")
    print("   1. 确认 KEYWORDS 列表是否需要增删。")
    print("   2. 将高频干扰节点 (如 'Blood', 'Cytoplasm' 等) 复制到下一步的 BLACKLIST_NODES。")

if __name__ == "__main__":
    if os.path.exists(KG_FILE):
        main()
    else:
        print(f"❌ 找不到文件: {KG_FILE}")

🚀 [Step 1] 正在加载 kg.csv 进行 NACC 适配侦察...
✅ PrimeKG 加载完成，共 8100498 条数据。

🔎 1. 关键词命中测试 (确认 NACC 特征能否映射到生物图谱)

--- 【核心-AD与认知】 ---
   ✅ 命中 8048 条相关关系
   📌 典型节点示例: ['taurine binding', 'Focal aware cognitive seizure with anomia', 'Focal cognitive seizure', 'Renal glomerular amyloid deposition', 'short-term memory']

--- 【NACC-心血管与代谢】 ---
   ✅ 命中 22666 条相关关系
   📌 典型节点示例: ['photomyoclonus, diabetes mellitus, deafness, nephropathy, and cerebral dysfunction', '25-hydroxycholesterol 7alpha-hydroxylase activity', 'NDP-glucose-starch glucosyltransferase activity', 'pulmonary hypertension', 'regulation of phospholipid efflux']

--- 【NACC-精神与其它】 ---
   ✅ 命中 41866 条相关关系
   📌 典型节点示例: ['Hypothalamic hypothyroidism', 'Hyperparathyroidism', 'thyroiditis (disease)', 'parenchyma of thyroid gland', 'Thyroid C cell hyperplasia']

💣 2. 黑名单侦察 (找出连接数过高且无特异性的干扰节点)
   说明：以下节点如果在 Top 20 且是通用解剖部位(如 kidney, lung)或通用概念，
   务必添加到下一步的 BLACKLIST_NODES 中！

Rank  | Node Name                                | Count     
------

In [1]:
import pandas as pd
import os

# NACC 文件列表
NACC_FILES = ['NACC_ad.csv', 'NACC_mci.csv', 'NACC_normal.csv']

# 病史字段 
HIS_COLS = {
    "Hypertension (高血压)": "his_HYPERTEN",
    "Hypercholesterolemia (高血脂)": "his_HYPERCHO", 
    "Diabetes (糖尿病)": "his_DIABETES",
    "Thyroid (甲状腺)": "his_THYROID",
    "Stroke (中风)": "his_CBSTROKE",
    "Heart Failure (心衰)": "his_CVCHF",
    "Atrial Fib (房颤)": "his_CVAFIB",
    "Depression (抑郁)": "his_DEP2YRS",
    "Anxiety (焦虑)": "his_ANXIETY"
}

def analyze_nacc_prevalence():
    print("📊 正在统计 NACC 临床数据中的病史出现频率...")
    
    total_patients = 0
    stats = {k: 0 for k in HIS_COLS.keys()}
    
    for fname in NACC_FILES:
        if not os.path.exists(fname):
            continue
            
        try:
            df = pd.read_csv(fname)
            total_patients += len(df)
            
            for label, col in HIS_COLS.items():
                if col in df.columns:
                    # NACC 编码通常是: 1=Yes, 0=No, 有时候 2=Unknown
                    # 我们只统计明确为 1 (Yes) 的
                    count = df[col].apply(lambda x: 1 if x == 1 else 0).sum()
                    stats[label] += count
        except Exception as e:
            print(f"⚠️ 读取 {fname} 出错: {e}")

    print(f"\n👥 总样本量: {total_patients}")
    print("-" * 50)
    print(f"{'特征':<25} | {'患病人数':<10} | {'占比 (%)':<10}")
    print("-" * 50)
    
    # 排序并打印
    sorted_stats = sorted(stats.items(), key=lambda x: x[1], reverse=True)
    
    for label, count in sorted_stats:
        ratio = (count / total_patients) * 100
        print(f"{label:<25} | {count:<10} | {ratio:.1f}%")
        
    print("-" * 50)
    print("💡 决策建议：")
    print("   - 占比 < 5% 的特征，建议从知识图谱中【完全剔除】，因为样本太少无法训练。")
    print("   - 占比 > 20% 的特征，是核心驱动力，必须保留其复杂的生物学机制。")

analyze_nacc_prevalence()

📊 正在统计 NACC 临床数据中的病史出现频率...

👥 总样本量: 3708
--------------------------------------------------
特征                        | 患病人数       | 占比 (%)    
--------------------------------------------------
Hypercholesterolemia (高血脂) | 1553       | 41.9%
Hypertension (高血压)        | 1388       | 37.4%
Depression (抑郁)           | 753        | 20.3%
Thyroid (甲状腺)             | 503        | 13.6%
Diabetes (糖尿病)            | 381        | 10.3%
Anxiety (焦虑)              | 230        | 6.2%
Atrial Fib (房颤)           | 120        | 3.2%
Heart Failure (心衰)        | 30         | 0.8%
Stroke (中风)               | 28         | 0.8%
--------------------------------------------------
💡 决策建议：
   - 占比 < 5% 的特征，建议从知识图谱中【完全剔除】，因为样本太少无法训练。
   - 占比 > 20% 的特征，是核心驱动力，必须保留其复杂的生物学机制。
